# Дообучение Qwen3.5-4B (LoRA SFT) в Colab

Ноутбук запускает тот же `scripts/train_sft.py`, что и основной путь на RTX 5070 (см. `docs/finetune.md`,
раздел «Обучение и возврат в LM Studio»).

- **T4 — только пробный запуск.** На T4 (sm_75) Qwen3.5 учится во float32: в fp16 появляются NaN,
  поэтому Unsloth сам переходит на float32, и это примерно в 10 раз медленнее. Полное обучение — на RTX 5070 (bf16)
  или в Colab на A100/L4.
- **Данные не публикуются.** Датасет траекторий получен через API Mercury 3 (Inception): только внутреннее
  использование в курсовом проекте. Файлы лежат в вашем личном Google Drive; ноутбук с выводом ячеек не выкладывать.
- Ключи API здесь не нужны: модель скачивается с Hugging Face без токена.

## 1. Видеокарта

Проверяем GPU. На T4 скрипт сам выберет float32 и предупредит; тогда пропустите раздел 6 (полное обучение).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2. Установка (стек Unsloth для Qwen3.5)

Проверенный набор из официального ноутбука Unsloth `Qwen3_5_(4B)_Vision.ipynb`: torch 2.8 (cu128),
transformers 5.2.0, trl 0.22.2, unsloth и unsloth_zoo из git, затем ядра. Если установка конфликтует,
скопируйте ячейку установки из актуальной версии того ноутбука — она первоисточник.
После установки перезапустите среду (Runtime → Restart session).

In [ ]:
%%capture
!pip install -q "torch==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128
!pip install -q "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git" "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "transformers==5.2.0" "trl==0.22.2" peft accelerate datasets bitsandbytes
!unsloth install-kernels

In [ ]:
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| sm", torch.cuda.get_device_capability(0),
      "| bf16:", torch.cuda.is_bf16_supported())
if torch.cuda.get_device_capability(0)[0] < 8:
    print("ВНИМАНИЕ: T4 — Qwen3.5 будет учиться во float32 (~10x медленнее). Только пробный запуск.")

## 3. Google Drive и файлы

Заранее положите в папку `research-assistant-ft` на своём Drive:

- `train_sft.py` — из `scripts/` репозитория;
- `qwen3_5_chat_template.jinja` — из `src/research_assistant/finetune/`;
- `sft/train.jsonl`, `sft/val.jsonl` — из `data/finetune/sft/` (результат `ft-export`).

Вместо Drive можно загрузить файлы ячейкой с `files.upload()` ниже, но тогда результаты пропадут вместе со средой.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORK = "/content/drive/MyDrive/research-assistant-ft"   # данные и результаты на Drive
LOCAL = "/content/ft"                                   # рабочая копия на диске среды

In [ ]:
import os, shutil
os.makedirs(f"{LOCAL}/sft", exist_ok=True)
for name in ["train_sft.py", "qwen3_5_chat_template.jinja", "sft/train.jsonl", "sft/val.jsonl"]:
    src = f"{WORK}/{name}"
    if os.path.exists(src):
        shutil.copy(src, f"{LOCAL}/{name}")
        print("скопирован", name, os.path.getsize(src) // 1024, "КБ")
    else:
        print("НЕТ ФАЙЛА", src)

In [ ]:
# Запасной путь без Drive: загрузить файлы вручную (раскомментируйте)
# from google.colab import files
# for name, data in files.upload().items():
#     target = f"{LOCAL}/sft/{name}" if name.endswith(".jsonl") else f"{LOCAL}/{name}"
#     open(target, "wb").write(data)

## 4. Длины примеров (без обучения)

`--dry-run` только токенизирует и печатает распределение длин, долю обучаемых токенов и пример участков
с потерями. По нему выбирается `--max-len`: примеры длиннее отбрасываются, а не режутся посреди хода.

In [ ]:
%cd {LOCAL}
!python train_sft.py --train sft/train.jsonl --val sft/val.jsonl --max-len 4096 --dry-run \
    --chat-template qwen3_5_chat_template.jinja

## 5. Пробный запуск: 30 шагов

Проверяет, что модель грузится, LoRA встаёт на проекции Gated DeltaNet (`in_proj_qkv`, `in_proj_z`,
`in_proj_a`, `in_proj_b`, `out_proj`) и на q/k/v/o/MLP, loss падает и нет NaN.

`--max-len 4096`: один только системный промпт со схемами инструментов — около 3000 токенов, поэтому
при 2048 не остаётся ни одного примера. Если в ячейке 4 при 4096 отброшено больше половины train,
возьмите длину не меньше p50 оттуда. `--limit-train 200` держит токенизацию на T4 короткой.

In [ ]:
OUT = f"{WORK}/runs/smoke"
!python train_sft.py --train sft/train.jsonl --val sft/val.jsonl --limit-train 200 --limit-val 20 \
    --max-len 4096 --max-steps 30 --eval-steps 10 --save-steps 10 --logging-steps 1 \
    --out {OUT} --chat-template qwen3_5_chat_template.jinja

## 6. Полное обучение (не на T4)

Только на L4 или A100 (bf16). Ячейка переопределяет `OUT`, поэтому слияние ниже возьмёт полный, а не пробный адаптер.

Две эпохи, lr 1e-4, косинус, эффективный батч 16. Лучший чекпоинт по eval loss сохраняется как `adapter/`;
окончательный выбор — по метрикам агента (`eval-agent`). Обрыв среды не страшен: `--resume` продолжит
с последнего чекпоинта на Drive.

In [ ]:
# Не запускайте на T4. Продолжить после обрыва: та же команда с --resume
OUT = f"{WORK}/runs/r16"
!python train_sft.py --train sft/train.jsonl --val sft/val.jsonl \
    --max-len 8192 --epochs 2 --lr 1e-4 --rank 16 --alpha 16 --batch 1 --grad-accum 16 \
    --eval-steps 50 --save-steps 50 --out {OUT} --chat-template qwen3_5_chat_template.jinja

## 7. Слияние и GGUF (по желанию)

Слитая 16-битная модель (~9 ГБ) плюс шаблон LM Studio, затем свежий llama.cpp (поддержка qwen35 — с февраля 2026):
`convert_hf_to_gguf.py` → bf16 GGUF → `llama-quantize` Q4_K_M (~2.7 ГБ). Последняя ячейка сверяет чат-шаблон
внутри GGUF с шаблоном LM Studio.

In [ ]:
MERGED = f"{LOCAL}/merged"
!python train_sft.py --merge-from {OUT}/adapter --merge-out {MERGED} --chat-template qwen3_5_chat_template.jinja

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null

In [ ]:
GGUF_BF16 = f"{LOCAL}/qwen3.5-4b-ra-bf16.gguf"
GGUF_Q4 = f"{WORK}/qwen3.5-4b-ra-Q4_K_M.gguf"
!python llama.cpp/convert_hf_to_gguf.py {MERGED} --outfile {GGUF_BF16} --outtype bf16
!llama.cpp/build/bin/llama-quantize {GGUF_BF16} {GGUF_Q4} Q4_K_M
!pip install -q gguf
!python {LOCAL}/train_sft.py --check-gguf {GGUF_Q4} --chat-template {LOCAL}/qwen3_5_chat_template.jinja

## 8. Скачивание и возврат в LM Studio

1. Скачайте `qwen3.5-4b-ra-Q4_K_M.gguf` с Drive через браузер (через `files.download` большие файлы рвутся).
   Адаптер `runs/.../adapter` (~100 МБ) тоже стоит сохранить.
2. На Windows:

   ```
   lms import C:\путь\к\qwen3.5-4b-ra-Q4_K_M.gguf
   lms load qwen3.5-4b-ra --context-length 16384 --gpu max -y
   ```

3. Оценка: `eval-agent --variant prefetch --runs 3` с `MODEL=<id новой модели>` — команды в `docs/finetune.md`.

Модель и датасет не публикуются (условия Inception: только внутреннее использование).